# Experiment 1: agreement counts and Qwen candidate logits

This notebook sweeps 11 `XVsYPosteriorProbe.reasoning_budget` values (`0, 20, ..., 200`) over the same `num_question_sets * 2**K` exhaustive noisy-channel evidence scenarios **per model**. With `CONTROL_POSITIONAL_BIAS=True`, every scenario is presented twice: once as $C_1$ then $C_2$, and once as $C_2$ then $C_1$. Thus each model-specific dataset has exactly $11 \times 8 \times 10 \times 2 = 1760$ inference rows.

`MetricSpec` identifies the candidate suffix surfaces (`2` and `7`) and their token IDs. At budget 0 the assistant prefill is exactly `ANSWER:`. At positive budgets, the enforced `REASONING:` continuation and `ANSWER:` are one continued assistant prefill with no intervening user turn. This notebook stores only the two unprocessed candidate logits emitted by the first answer-generation step; it does not run a duplicate answer-boundary forward pass or persist full-vocabulary tensors. Full-vocabulary and all-stream/all-layer/all-position capture remain available through `CaptureSpec`. Canonical $C_1/C_2$ metrics are paired and averaged across presentation order before grouping.

In [1]:
from __future__ import annotations

import logging
import os
import threading

os.environ.setdefault("TORCHAO_FORCE_SKIP_LOADING_SO_FILES", "1")
logging.getLogger("torchao").setLevel(logging.ERROR)

import csv
import gc
import hashlib
import html
import math
import re
import sys
from collections import defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from statistics import correlation, fmean, pstdev
from time import monotonic

import ipywidgets as widgets
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display
from transformers import AutoProcessor

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(REPO_ROOT))

# All experiment-framework imports come from noisy_channel_bayesian.
from mats_experiments.noisy_channel_bayesian import (
    CaptureSpec,
    ExecutionConfig,
    MetricSpec,
    ModelConfig,
    NoisyChannelBayesianEnvironment,
    QwenRunner,
    RandomSubsetQuestion,
    SystemPrompt,
    TokenizerBinding,
    TranscriptDataset,
    TranscriptDatasetGenerator,
    XVsYPosteriorProbe,
    answer_patterns,
    get_answer_surface_logits,
)

## Configuration

`MODEL_SELECTION` accepts either one model ID string or a list of model ID strings. Every model gets a separately tokenized dataset and a separate run directory, while the fixed seed keeps its sampled question banks identical across models.

In [13]:
# Requested experiment constants.
N = 8
K = 3
R_VALUES = "9/10"
SYSTEM_PROMPT_TEXT = (
    "You are a bayesian reasoner. Follow the user's game rules exactly."
)
NUM_QUESTION_SETS = 10
CONTROL_POSITIONAL_BIAS = True
# Eleven values: Python's stop is exclusive, so this is 0, 20, ..., 200.
REASONING_BUDGET_VALUES = tuple(range(0, 201, 20))

# Use a string for one model, or e.g. ["Qwen/Qwen3.5-4B", "Qwen/Qwen3.5-9B"].
MODEL_SELECTION: str | list[str] = ["Qwen/Qwen3.5-4B", "Qwen/Qwen3.5-9B"]

# The requested fixed probe and question construction.
X = 2
Y = 7
SUBSET_SIZE = 4
SEED = 20260902

# Execution settings.
MODEL_DTYPE = "auto"
DEVICE_MAP = None  # None lets QwenRunner choose "auto" when CUDA is available.
# Stage-specific defaults for a 32 GiB GPU. OOM recovery still splits minibatches.
STAGE_BATCH_SIZES_BY_MODEL = {
    "Qwen/Qwen3.5-4B": {
        "reasoning": 40, "answer": 40, "capture": 16, "score": 16,
    },
    "Qwen/Qwen3.5-9B": {
        "reasoning": 24, "answer": 24, "capture": 8, "score": 8,
    },
}
DEFAULT_STAGE_BATCH_SIZES = {
    "reasoning": 4, "answer": 4, "capture": 2, "score": 4,
}
CHECKPOINT_EVERY_BATCHES = 25
LOG_INTERVAL_SECONDS = 30
MAX_ANSWER_TOKENS = 4
MAX_REASONING_TOKENS = 256
LOCAL_FILES_ONLY = True
RUN_MODEL = True
EXPERIMENT_ROOT = REPO_ROOT / "artifacts" / "noisy_channel_bayesian_experiment_1"

In [14]:
def log_progress(message: str) -> None:
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    print(f"[{timestamp}] {message}", flush=True)


def checkpoint_row_count(path: Path) -> int:
    if not path.exists():
        return 0
    try:
        with path.open("r", encoding="utf-8") as handle:
            return sum(bool(line.strip()) for line in handle)
    except OSError:
        # The runner atomically replaces this file at periodic answer checkpoints.
        return 0


@contextmanager
def progress_heartbeat(label: str, checkpoint_path: Path | None = None):
    started = monotonic()
    stopped = threading.Event()

    def report() -> None:
        while not stopped.wait(LOG_INTERVAL_SECONDS):
            elapsed = monotonic() - started
            suffix = ""
            if checkpoint_path is not None:
                completed = checkpoint_row_count(checkpoint_path)
                suffix = (
                    f"; answer checkpoints={completed}/"
                    f"{EXPECTED_INFERENCE_ROWS_PER_MODEL}"
                )
            log_progress(f"{label}: still running; elapsed={elapsed / 60:.1f} min{suffix}")

    log_progress(f"{label}: started")
    worker = threading.Thread(target=report, name=f"{label}-heartbeat", daemon=True)
    worker.start()
    succeeded = False
    try:
        yield
        succeeded = True
    finally:
        stopped.set()
        worker.join(timeout=1)
        status = "finished" if succeeded else "stopped with an error"
        log_progress(f"{label}: {status} after {(monotonic() - started) / 60:.1f} min")


def normalize_model_ids(selection: str | list[str]) -> list[str]:
    if isinstance(selection, str):
        model_ids = [selection]
    elif isinstance(selection, list) and all(isinstance(item, str) for item in selection):
        model_ids = list(selection)
    else:
        raise TypeError("MODEL_SELECTION must be a model ID string or a list of strings.")
    if not model_ids or any(not item.strip() for item in model_ids):
        raise ValueError("At least one non-empty model ID is required.")
    if len(set(model_ids)) != len(model_ids):
        raise ValueError("MODEL_SELECTION contains a duplicate model ID.")
    return model_ids


def model_key(model_id: str) -> str:
    readable = re.sub(r"[^A-Za-z0-9_.-]+", "_", model_id).strip("_")
    digest = hashlib.sha256(model_id.encode()).hexdigest()[:8]
    return f"{readable}_{digest}"


MODEL_IDS = normalize_model_ids(MODEL_SELECTION)
environment = NoisyChannelBayesianEnvironment(
    n=N, k=K, r_values=R_VALUES, control_positional_bias=CONTROL_POSITIONAL_BIAS
)
question = RandomSubsetQuestion(subset_size=SUBSET_SIZE, replacement=False, sort=True)
probes = tuple(
    XVsYPosteriorProbe(
        x=X,
        y=Y,
        reasoning_budget=reasoning_budget,
        allow_same=False,
        call_layout="conversation",
    )
    for reasoning_budget in REASONING_BUDGET_VALUES
)
system_prompt = SystemPrompt(SYSTEM_PROMPT_TEXT)

# MetricSpec maps X/Y to their model-facing token surfaces. CaptureSpec stores raw logits.
# Sequence scoring is disabled because the requested quantities are raw logits, not log-probs.
metric_spec = MetricSpec(sequence_scores=False)
capture_spec = CaptureSpec(
    logits_boundaries=("answer",),
    logits_scope="answer_surfaces",
    streams=(),
    every_decode_position=False,
)

assert environment.shared_reliability
assert len(set(environment.reliabilities)) == 1
assert 0 < environment.reliabilities[0] < 1
assert environment.control_positional_bias is CONTROL_POSITIONAL_BIAS
assert CONTROL_POSITIONAL_BIAS is True
assert len(REASONING_BUDGET_VALUES) == 11
assert REASONING_BUDGET_VALUES == tuple(range(0, 201, 20))
assert tuple(probe.reasoning_budget for probe in probes) == REASONING_BUDGET_VALUES
assert all((probe.x, probe.y, probe.allow_same) == (2, 7, False) for probe in probes)
assert MAX_REASONING_TOKENS >= max(REASONING_BUDGET_VALUES)
EXPECTED_INFERENCE_ROWS_PER_MODEL = (
    len(REASONING_BUDGET_VALUES) * 2**K * NUM_QUESTION_SETS * 2
)
assert EXPECTED_INFERENCE_ROWS_PER_MODEL == 1760
print({
    "models": MODEL_IDS,
    "reasoning_budget_values": REASONING_BUDGET_VALUES,
    "evidence_scenarios_per_reasoning_budget": NUM_QUESTION_SETS * 2**K,
    "inference_rows_per_model": EXPECTED_INFERENCE_ROWS_PER_MODEL,
    "stage_batch_sizes": {
        model_id: STAGE_BATCH_SIZES_BY_MODEL.get(
            model_id, DEFAULT_STAGE_BATCH_SIZES
        )
        for model_id in MODEL_IDS
    },
    "reliability": str(environment.reliabilities[0]),
    "artifact_root": str(EXPERIMENT_ROOT),
})

{'models': ['Qwen/Qwen3.5-4B', 'Qwen/Qwen3.5-9B'], 'reasoning_budget_values': (0, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200), 'evidence_scenarios_per_reasoning_budget': 80, 'inference_rows_per_model': 1760, 'stage_batch_sizes': {'Qwen/Qwen3.5-4B': {'reasoning': 48, 'answer': 48, 'capture': 16, 'score': 16}, 'Qwen/Qwen3.5-9B': {'reasoning': 24, 'answer': 24, 'capture': 8, 'score': 8}}, 'reliability': '9/10', 'artifact_root': '/workspace/MATS/artifacts/noisy_channel_bayesian_experiment_1'}


## Agreement fields

For candidate $c$ and question $i$, agreement is 1 exactly when the observed report equals the YES/NO membership answer predicted by $c$:

$$a_i^j(c)=\mathbf{1}\{\text{report}_i^j = (\text{YES if }c\in Q_i^j\text{ else NO})\}. $$

The total $a^j(c)=\sum_i a_i^j(c)$ is stored alongside every generated transcript before model execution. `candidate_1` and `candidate_2` are stable semantic identities; `x` and `y` mean first and second in the current presentation and therefore swap.

In [15]:
def validate_exhaustive_dataset(dataset: TranscriptDataset) -> None:
    expected_patterns = ["".join("Y" if value == "YES" else "N" for value in pattern)
                         for pattern in answer_patterns(K)]
    assert len(dataset) == EXPECTED_INFERENCE_ROWS_PER_MODEL
    assert dataset.manifest["parameterization_count"] == len(REASONING_BUDGET_VALUES)
    assert dataset.manifest["environment_parameter_count"] == 1
    assert dataset.manifest["question_parameter_count"] == 1
    assert dataset.manifest["probe_parameter_count"] == len(REASONING_BUDGET_VALUES)
    assert tuple(dataset.manifest["reasoning_budgets"]) == REASONING_BUDGET_VALUES
    assert dataset.manifest["num_question_sets"] == NUM_QUESTION_SETS
    assert dataset.manifest["presentations_per_scenario"] == 2
    assert dataset.manifest["control_positional_bias"] is True
    assert all(row["n"] == N and row["k"] == K for row in dataset)
    assert all(row["candidate_1"] == X and row["candidate_2"] == Y for row in dataset)
    assert all(row["shared_reliability"] for row in dataset)
    expected_reliabilities = [str(value) for value in environment.reliabilities]
    assert all(row["reliabilities_exact"] == expected_reliabilities for row in dataset)
    assert all(
        row["messages"][0] == {"role": "system", "content": SYSTEM_PROMPT_TEXT}
        for row in dataset
    )
    schedules_by_budget = {}
    for probe_index, reasoning_budget in enumerate(REASONING_BUDGET_VALUES):
        budget_rows = [row for row in dataset if row["reasoning_budget"] == reasoning_budget]
        assert len(budget_rows) == NUM_QUESTION_SETS * 2**K * 2
        assert all(row["probe_parameter_index"] == probe_index for row in budget_rows)
        schedules_by_budget[reasoning_budget] = question_schedules(dataset, reasoning_budget)
        for question_set_index in range(NUM_QUESTION_SETS):
            rows = [
                row for row in budget_rows
                if row["question_set_index"] == question_set_index
            ]
            assert [row["answer_pattern"] for row in rows] == [
                pattern for pattern in expected_patterns for _ in range(2)
            ]
            assert len({tuple(map(tuple, row["membership_sets"])) for row in rows}) == 1
            pairs = defaultdict(list)
            for row in rows:
                pairs[row["positional_control_pair_id"]].append(row)
            assert len(pairs) == 2**K
            for pair in pairs.values():
                assert [row["presentation_order"] for row in pair] == ["C1_C2", "C2_C1"]
                assert [(row["x"], row["y"]) for row in pair] == [(X, Y), (Y, X)]
                for invariant in (
                    "reasoning_budget", "membership_sets", "observed_reports",
                    "prior_predictive_exact", "posterior_exact",
                    "agreement_candidate_1_by_question",
                    "agreement_candidate_2_by_question",
                ):
                    assert pair[0][invariant] == pair[1][invariant]
    assert all(
        schedules == schedules_by_budget[REASONING_BUDGET_VALUES[0]]
        for schedules in schedules_by_budget.values()
    ), "Question banks changed across reasoning budgets."
    for row in dataset:
        assert len(row["agreement_x_by_question"]) == K
        assert len(row["agreement_y_by_question"]) == K
        assert row["total_agreement_x"] == sum(row["agreement_x_by_question"])
        assert row["total_agreement_y"] == sum(row["agreement_y_by_question"])
        assert row["total_agreement_candidate_1"] == sum(
            row["agreement_candidate_1_by_question"]
        )
        assert row["total_agreement_candidate_2"] == sum(
            row["agreement_candidate_2_by_question"]
        )


def question_schedules(
    dataset: TranscriptDataset, reasoning_budget: int
) -> list[list[list[int]]]:
    return [
        next(
            row for row in dataset
            if row["question_set_index"] == index
            and row["reasoning_budget"] == reasoning_budget
        )["membership_sets"]
        for index in range(NUM_QUESTION_SETS)
    ]

## Generate, execute, and read the raw candidate logits

A model-specific `AutoProcessor` supplies the construction tokenizer. `QwenRunner` then loads the same model ID and reserializes with its runtime tokenizer. The equality assertion below verifies that both chat-template fingerprints match.

For each result, `get_answer_surface_logits` reads the two singleton candidate-token logits returned by the first answer-generation step. These are raw, pre-softmax logits—not continuation log-probabilities. Set `logits_scope="full"` to persist full-vocabulary tensors instead; `logit_tokens="all"` captures every prompt position, while `every_decode_position=True` captures every generated position. Activation capture remains independently configurable with all streams, all layers, and `tokens="all"`.

The execution cell emits timestamped stage messages and a heartbeat every `LOG_INTERVAL_SECONDS`. Answer checkpoint counts remain zero while the runner completes all positive-budget reasoning generations, then advance every `CHECKPOINT_EVERY_BATCHES` answer minibatches. Reasoning, answer generation, capture, and scoring have independent model-specific batch sizes; the runner recursively splits an individual generation, capture, or scoring minibatch if CUDA reports OOM.

In [16]:
datasets_by_model: dict[str, TranscriptDataset] = {}
results_by_model: dict[str, list[dict[str, object]]] = {}
balanced_results_by_model: dict[str, list[dict[str, object]]] = {}
run_dirs_by_model: dict[str, Path] = {}
all_result_rows: list[dict[str, object]] = []
reference_schedules = None


def balance_positional_pairs(
    rows: list[dict[str, object]],
) -> list[dict[str, object]]:
    pairs = defaultdict(list)
    for row in rows:
        pairs[row["positional_control_pair_id"]].append(row)

    balanced_rows = []
    invariant_fields = (
        "reasoning_budget", "probe_parameter_index",
        "question_set_index", "answer_pattern_index", "answer_pattern",
        "membership_sets", "observed_reports", "reliabilities_exact",
        "prior_predictive_exact", "posterior_exact",
        "candidate_1_posterior_exact", "candidate_2_posterior_exact",
        "candidate_1_minus_candidate_2_log_odds",
        "agreement_candidate_1_by_question",
        "agreement_candidate_2_by_question",
        "total_agreement_candidate_1", "total_agreement_candidate_2",
    )
    for pair_id, pair in pairs.items():
        by_order = {row["presentation_order"]: row for row in pair}
        assert set(by_order) == {"C1_C2", "C2_C1"}
        first = by_order["C1_C2"]
        second = by_order["C2_C1"]
        for field in invariant_fields:
            assert first[field] == second[field], (pair_id, field)

        difference_first = float(first["candidate_1_minus_candidate_2_raw_logit"])
        difference_second = float(second["candidate_1_minus_candidate_2_raw_logit"])
        balanced = dict(first)
        balanced.update({
            "balanced_across_presentation_order": True,
            "candidate_1_raw_logit_order_c1_c2": float(first["candidate_1_raw_logit"]),
            "candidate_1_raw_logit_order_c2_c1": float(second["candidate_1_raw_logit"]),
            "candidate_2_raw_logit_order_c1_c2": float(first["candidate_2_raw_logit"]),
            "candidate_2_raw_logit_order_c2_c1": float(second["candidate_2_raw_logit"]),
            "candidate_1_raw_logit": fmean([
                float(first["candidate_1_raw_logit"]),
                float(second["candidate_1_raw_logit"]),
            ]),
            "candidate_2_raw_logit": fmean([
                float(first["candidate_2_raw_logit"]),
                float(second["candidate_2_raw_logit"]),
            ]),
            "candidate_1_minus_candidate_2_raw_logit": fmean([
                difference_first, difference_second
            ]),
            "position_effect_on_candidate_difference": 0.5 * (
                difference_first - difference_second
            ),
            "candidate_1_first_minus_second_raw_logit": (
                float(first["candidate_1_raw_logit"])
                - float(second["candidate_1_raw_logit"])
            ),
            "candidate_2_first_minus_second_raw_logit": (
                float(second["candidate_2_raw_logit"])
                - float(first["candidate_2_raw_logit"])
            ),
            "model_choice_canonical_by_order": {
                order: row["model_choice_canonical"] for order, row in by_order.items()
            },
            "posterior_correct_by_order": {
                order: row["posterior_correct"] for order, row in by_order.items()
            },
            # Tensor captures remain presentation-specific and are never averaged.
            "presentation_artifacts": {
                order: {
                    key: row.get(key)
                    for key in ("row_id", "logit_path", "activation_path")
                }
                for order, row in by_order.items()
            },
        })
        balanced_rows.append(balanced)

    balanced_rows.sort(key=lambda row: (
        int(row["reasoning_budget"]),
        int(row["question_set_index"]),
        int(row["answer_pattern_index"]),
    ))
    assert len(balanced_rows) == len(REASONING_BUDGET_VALUES) * NUM_QUESTION_SETS * 2**K
    return balanced_rows

run_started = monotonic()
log_progress(
    f"Experiment started: {len(MODEL_IDS)} model(s), "
    f"{EXPECTED_INFERENCE_ROWS_PER_MODEL} rows/model"
)
for model_index, model_id in enumerate(MODEL_IDS, start=1):
    model_started = monotonic()
    model_batch_sizes = STAGE_BATCH_SIZES_BY_MODEL.get(
        model_id, DEFAULT_STAGE_BATCH_SIZES
    )
    log_progress(
        f"Model {model_index}/{len(MODEL_IDS)} {model_id}: loading tokenizer; "
        f"stage_batch_sizes={model_batch_sizes}"
    )
    processor = AutoProcessor.from_pretrained(
        model_id,
        local_files_only=LOCAL_FILES_ONLY,
    )
    tokenizer = getattr(processor, "tokenizer", processor)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer_binding = TokenizerBinding(tokenizer, enable_thinking=False)

    dataset_started = monotonic()
    log_progress(f"{model_id}: generating and tokenizing transcript dataset")
    with progress_heartbeat(f"{model_id} dataset generation"):
        dataset = TranscriptDatasetGenerator(
            environment=environment,
            question=question,
            probe=probes,
            tokenizer_binding=tokenizer_binding,
            system_prompt=system_prompt,
            seed=SEED,
        ).generate(num_question_sets=NUM_QUESTION_SETS)
    validate_exhaustive_dataset(dataset)
    log_progress(
        f"{model_id}: validated {len(dataset)} rows in "
        f"{monotonic() - dataset_started:.1f} s"
    )

    schedules = question_schedules(dataset, REASONING_BUDGET_VALUES[0])
    if reference_schedules is None:
        reference_schedules = schedules
    else:
        assert schedules == reference_schedules, "Question banks changed across models."

    experiment_dir = EXPERIMENT_ROOT / model_key(model_id)
    dataset.save(experiment_dir)
    log_progress(f"{model_id}: dataset saved to {experiment_dir}")
    datasets_by_model[model_id] = dataset

    if not RUN_MODEL:
        log_progress(f"RUN_MODEL=False: generated {len(dataset)} rows but skipped {model_id}.")
        del processor, tokenizer
        continue

    execution = ExecutionConfig(
        experiment_dir=experiment_dir,
        run_id="agreement_raw_logits_reasoning_sweep_positional_control_v2_optimized",
        batch_size=DEFAULT_STAGE_BATCH_SIZES["answer"],
        reasoning_batch_size=model_batch_sizes["reasoning"],
        answer_batch_size=model_batch_sizes["answer"],
        capture_batch_size=model_batch_sizes["capture"],
        score_batch_size=model_batch_sizes["score"],
        checkpoint_every_batches=CHECKPOINT_EVERY_BATCHES,
        max_answer_tokens=MAX_ANSWER_TOKENS,
        max_reasoning_tokens=MAX_REASONING_TOKENS,
        resume=True,
        metrics=metric_spec,
        capture=capture_spec,
    )
    runner = QwenRunner(ModelConfig(
        model_name_or_path=model_id,
        dtype=MODEL_DTYPE,
        device_map=DEVICE_MAP,
        local_files_only=LOCAL_FILES_ONLY,
    ))
    run_dir = experiment_dir / "runs" / execution.run_id
    results_path = run_dir / "results.jsonl"
    run_dirs_by_model[model_id] = run_dir
    resumed_rows = checkpoint_row_count(results_path) if execution.resume else 0
    log_progress(
        f"{model_id}: starting inference with {resumed_rows} resumed answer rows. "
        "The checkpoint stays at zero until the positive-budget reasoning stage finishes."
    )
    with progress_heartbeat(f"{model_id} inference", results_path):
        results = dataset.execute(runner, execution)

    enriched_results = []
    log_progress(f"{model_id}: enriching {len(results)} inference rows")
    for result_index, result in enumerate(results, start=1):
        reasoning_budget = int(result["reasoning_budget"])
        assert reasoning_budget in REASONING_BUDGET_VALUES
        if reasoning_budget == 0:
            assert result["enforced_reasoning"] is None
            assert result["zero_reasoning_compliance"] is True
            assert result["answer_messages"][-1] == {
                "role": "assistant", "content": result["answer_prefix"]
            }
        else:
            assert result["enforced_reasoning"] is not None
            assert result["zero_reasoning_compliance"] is None
            assert result["messages"][-1]["content"].endswith("REASONING:")
            assert result["answer_messages"][-1] == {
                "role": "assistant",
                "content": f"{result['enforced_reasoning'].rstrip()}\n{result['answer_prefix']}",
            }
        assert result["strict_answer_compliance"] is True
        allowed_completion = rf"ANSWER:\s*({re.escape(str(X))}|{re.escape(str(Y))})\s*"
        assert re.fullmatch(allowed_completion, result["assistant_completion"])
        assert result["answer_serialized_prompt"].endswith(result["answer_prefix"])
        assert result["runtime_tokenizer_template_fingerprint"] == result[
            "tokenizer_template_fingerprint"
        ]
        token_ids = result["answer_surface_token_ids"]
        if len(token_ids["X"]) != 1 or len(token_ids["Y"]) != 1:
            raise ValueError(
                f"{model_id} does not represent {X!r} and {Y!r} as singleton answer tokens: "
                f"{token_ids}"
            )
        surface_logits = get_answer_surface_logits(result, run_dir, boundary="answer")
        row = {
            **result,
            "model_id": model_id,
            "x_raw_logit": surface_logits[str(result["x"])],
            "y_raw_logit": surface_logits[str(result["y"])],
            "candidate_1_raw_logit": surface_logits[str(X)],
            "candidate_2_raw_logit": surface_logits[str(Y)],
        }
        row["x_minus_y_raw_logit"] = row["x_raw_logit"] - row["y_raw_logit"]
        row["candidate_1_minus_candidate_2_raw_logit"] = (
            row["candidate_1_raw_logit"] - row["candidate_2_raw_logit"]
        )
        enriched_results.append(row)
        all_result_rows.append(row)
        if result_index % 500 == 0 or result_index == len(results):
            log_progress(f"{model_id}: enriched {result_index}/{len(results)} rows")
    results_by_model[model_id] = enriched_results
    balanced_results_by_model[model_id] = balance_positional_pairs(enriched_results)
    log_progress(
        f"{model_id}: built {len(balanced_results_by_model[model_id])} "
        "order-balanced scenarios"
    )

    sample = enriched_results[0]
    display({
        "model_id": model_id,
        "reasoning_budget_values": REASONING_BUDGET_VALUES,
        "inference_rows": len(enriched_results),
        "balanced_evidence_scenarios": len(balanced_results_by_model[model_id]),
        "user_prompt_ends_in": sample["messages"][-1]["content"][-120:],
        "assistant_completion": sample["assistant_completion"],
        "answer_prompt_ends_in": sample["answer_serialized_prompt"][-120:],
        "captured_prompt_position": len(sample["answer_input_ids"]) - 1,
        "last_serialized_input_token": sample["answer_input_tokens"][-1],
        "prediction_at_capture": "first generated answer token after the ANSWER: turn",
        "candidate_token_ids": {
            "X": sample["answer_surface_token_ids"]["X"],
            "Y": sample["answer_surface_token_ids"]["Y"],
        },
    })

    # Release each model before loading the next ID in a multi-model run.
    del runner, results, processor, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    log_progress(
        f"Model {model_index}/{len(MODEL_IDS)} {model_id}: complete in "
        f"{(monotonic() - model_started) / 60:.1f} min"
    )

if RUN_MODEL:
    assert len(all_result_rows) == len(MODEL_IDS) * EXPECTED_INFERENCE_ROWS_PER_MODEL
log_progress(f"Experiment cell complete in {(monotonic() - run_started) / 60:.1f} min")

# Print complete paired serializations to audit both candidate presentation orders.
log_progress("Printing full serialized prompt audit examples")
for model_id, rows in results_by_model.items():
    print(f"\n===== FULL SERIALIZED PROMPT EXAMPLES: {model_id} =====")
    pairs = defaultdict(list)
    for row in rows:
        pairs[row["positional_control_pair_id"]].append(row)
    pair_ids = list(pairs)
    for pair_id in (pair_ids[0], pair_ids[-1]):
        for row in pairs[pair_id]:
            print(
                f"\n--- pair={pair_id}; order={row['presentation_order']}; "
                f"budget={row['reasoning_budget']}; "
                f"question_set={row['question_set_index']}; pattern={row['answer_pattern']} ---"
            )
            print(row["answer_serialized_prompt"])
            print(f"ASSISTANT COMPLETION: {row['assistant_completion']!r}")
            print(
                "CANDIDATE TOKEN IDS: "
                f"X={row['answer_surface_token_ids']['X']}, "
                f"Y={row['answer_surface_token_ids']['Y']}"
            )

[2026-09-02 18:36:28 UTC] Experiment started: 2 model(s), 1760 rows/model
[2026-09-02 18:36:28 UTC] Model 1/2 Qwen/Qwen3.5-4B: loading tokenizer; stage_batch_sizes={'reasoning': 48, 'answer': 48, 'capture': 16, 'score': 16}
[2026-09-02 18:36:29 UTC] Qwen/Qwen3.5-4B: generating and tokenizing transcript dataset
[2026-09-02 18:36:29 UTC] Qwen/Qwen3.5-4B dataset generation: started
[2026-09-02 18:36:32 UTC] Qwen/Qwen3.5-4B dataset generation: finished after 0.0 min
[2026-09-02 18:36:32 UTC] Qwen/Qwen3.5-4B: validated 1760 rows in 2.6 s
[2026-09-02 18:36:32 UTC] Qwen/Qwen3.5-4B: dataset saved to /workspace/MATS/artifacts/noisy_channel_bayesian_experiment_1/Qwen_Qwen3.5-4B_227fdbeb
[2026-09-02 18:36:32 UTC] Qwen/Qwen3.5-4B: starting inference with 0 resumed answer rows. The checkpoint stays at zero until the positive-budget reasoning stage finishes.
[2026-09-02 18:36:32 UTC] Qwen/Qwen3.5-4B inference: started


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

[2026-09-02 18:37:02 UTC] Qwen/Qwen3.5-4B inference: still running; elapsed=0.5 min; answer checkpoints=0/1760
[2026-09-02 18:37:32 UTC] Qwen/Qwen3.5-4B inference: still running; elapsed=1.0 min; answer checkpoints=0/1760
[2026-09-02 18:38:02 UTC] Qwen/Qwen3.5-4B inference: still running; elapsed=1.5 min; answer checkpoints=0/1760
[2026-09-02 18:38:32 UTC] Qwen/Qwen3.5-4B inference: still running; elapsed=2.0 min; answer checkpoints=0/1760
[2026-09-02 18:39:02 UTC] Qwen/Qwen3.5-4B inference: still running; elapsed=2.5 min; answer checkpoints=0/1760
[2026-09-02 18:39:32 UTC] Qwen/Qwen3.5-4B inference: still running; elapsed=3.0 min; answer checkpoints=0/1760
[2026-09-02 18:40:02 UTC] Qwen/Qwen3.5-4B inference: still running; elapsed=3.5 min; answer checkpoints=0/1760
[2026-09-02 18:40:32 UTC] Qwen/Qwen3.5-4B inference: still running; elapsed=4.0 min; answer checkpoints=0/1760
[2026-09-02 18:41:02 UTC] Qwen/Qwen3.5-4B inference: still running; elapsed=4.5 min; answer checkpoints=0/1760
[

KeyboardInterrupt: 

## Aggregate raw logits by total agreement

For a candidate with $a$ agreements under shared reliability $r$, its exact unnormalized Bayesian log weight is

$$\log w(a)=-\log N+a\log r+(K-a)\log(1-r).$$

An individual language-model vocabulary logit has no uniquely determined Bayesian absolute level: a shared logit offset and token-specific surface bias do not affect the relevant candidate difference. Each candidate logit below is first averaged within its matched `C1_C2`/`C2_C1` pair. The primary difference statistic first computes $\ell(C_1)-\ell(C_2)$ in each prompt and then averages those two signed semantic differences. Consequently, the dashed candidate curves use the exact Bayesian slope $\log(r/(1-r))$ and fit one intercept per token surface.

$$\log\frac{w_{C_1}}{w_{C_2}}=(a(C_1)-a(C_2))\log\frac{r}{1-r}. $$

The matched position-effect estimate is half the difference between the semantic logit contrast when $C_1$ is first and when $C_1$ is second. Categorical completions and tensor activations are retained separately by order rather than averaged. Use the selector below to show the tables and charts for one fixed reasoning budget.

In [ ]:
RELIABILITY = float(environment.reliabilities[0])


def bayes_candidate_log_weight(total_agreement: int) -> float:
    return (
        -math.log(N)
        + total_agreement * math.log(RELIABILITY)
        + (K - total_agreement) * math.log1p(-RELIABILITY)
    )


def grouped_candidate_logits(
    rows: list[dict[str, object]], candidate_label: str
) -> list[dict[str, float | int]]:
    if candidate_label not in {"candidate_1", "candidate_2"}:
        raise ValueError("candidate_label must be 'candidate_1' or 'candidate_2'.")
    buckets: dict[int, list[float]] = defaultdict(list)
    for row in rows:
        agreement = int(row[f"total_agreement_{candidate_label}"])
        buckets[agreement].append(float(row[f"{candidate_label}_raw_logit"]))
    return [
        {
            "total_agreement": agreement,
            "mean_raw_logit": fmean(values),
            "count": len(values),
            "bayes_log_weight": bayes_candidate_log_weight(agreement),
        }
        for agreement, values in sorted(buckets.items())
    ]


def fitted_bayes_intercept(rows: list[dict[str, object]], candidate_label: str) -> float:
    return fmean(
        float(row[f"{candidate_label}_raw_logit"])
        - bayes_candidate_log_weight(int(row[f"total_agreement_{candidate_label}"]))
        for row in rows
    )


def positional_control_summary(
    raw_rows: list[dict[str, object]], balanced_rows: list[dict[str, object]]
) -> dict[str, object]:
    order_summaries = {}
    for order in ("C1_C2", "C2_C1"):
        order_rows = [row for row in raw_rows if row["presentation_order"] == order]
        eligible = [row for row in order_rows if row["posterior_correct"] is not None]
        order_summaries[order] = {
            "inference_count": len(order_rows),
            "mean_semantic_c1_minus_c2_logit": fmean(
                float(row["candidate_1_minus_candidate_2_raw_logit"])
                for row in order_rows
            ),
            "posterior_accuracy": (
                fmean(float(row["posterior_correct"]) for row in eligible)
                if eligible else None
            ),
            "strict_answer_compliance": fmean(
                float(row["strict_answer_compliance"]) for row in order_rows
            ),
            "canonical_choice_counts": {
                choice: sum(row["model_choice_canonical"] == choice for row in order_rows)
                for choice in ("C1", "C2", "SAME", None)
            },
        }
    agreement_differences = [
        int(row["total_agreement_candidate_1"])
        - int(row["total_agreement_candidate_2"])
        for row in balanced_rows
    ]
    balanced_differences = [
        float(row["candidate_1_minus_candidate_2_raw_logit"])
        for row in balanced_rows
    ]
    position_effects = [
        float(row["position_effect_on_candidate_difference"])
        for row in balanced_rows
    ]
    return {
        "by_presentation_order": order_summaries,
        "matched_scenario_count": len(balanced_rows),
        "balanced_agreement_logit_correlation": correlation(
            agreement_differences, balanced_differences
        ),
        "mean_position_effect_on_c1_minus_c2": fmean(position_effects),
        "sd_position_effect_on_c1_minus_c2": pstdev(position_effects),
    }


if not RUN_MODEL:
    raise RuntimeError("Set RUN_MODEL=True and execute the model cell before analysis.")


def rows_at_reasoning_budget(
    rows: list[dict[str, object]], reasoning_budget: int
) -> list[dict[str, object]]:
    selected = [row for row in rows if int(row["reasoning_budget"]) == reasoning_budget]
    assert selected, f"No rows found for reasoning_budget={reasoning_budget}."
    return selected


def reasoning_budget_selector(description: str) -> widgets.SelectionSlider:
    return widgets.SelectionSlider(
        options=REASONING_BUDGET_VALUES,
        value=REASONING_BUDGET_VALUES[0],
        description=description,
        continuous_update=False,
        readout=True,
        layout=widgets.Layout(width="720px"),
        style={"description_width": "140px"},
    )


def display_aggregate_tables(reasoning_budget: int) -> None:
    for model_id, all_balanced_rows in balanced_results_by_model.items():
        rows = rows_at_reasoning_budget(all_balanced_rows, reasoning_budget)
        raw_rows = rows_at_reasoning_budget(
            results_by_model[model_id], reasoning_budget
        )
        c1_groups = grouped_candidate_logits(rows, "candidate_1")
        c2_groups = grouped_candidate_logits(rows, "candidate_2")
        assert sum(group["count"] for group in c1_groups) == NUM_QUESTION_SETS * 2**K
        assert sum(group["count"] for group in c2_groups) == NUM_QUESTION_SETS * 2**K
        print(f"{model_id} — reasoning budget {reasoning_budget}")
        display({
            "positional control summary": positional_control_summary(raw_rows, rows),
            "C1 grouped values": c1_groups,
            "C2 grouped values": c2_groups,
        })


aggregate_budget_widget = reasoning_budget_selector("Reasoning budget")
aggregate_output = widgets.interactive_output(
    display_aggregate_tables, {"reasoning_budget": aggregate_budget_widget}
)
display(widgets.VBox([aggregate_budget_widget, aggregate_output]))

In [ ]:
def plot_agreement_logits(reasoning_budget: int) -> None:
    for model_id, all_rows in balanced_results_by_model.items():
        rows = rows_at_reasoning_budget(all_rows, reasoning_budget)
        c1_groups = grouped_candidate_logits(rows, "candidate_1")
        c2_groups = grouped_candidate_logits(rows, "candidate_2")
        c1_intercept = fitted_bayes_intercept(rows, "candidate_1")
        c2_intercept = fitted_bayes_intercept(rows, "candidate_2")

        fig, (ax_agreement, ax_difference, ax_position) = plt.subplots(
            1, 3, figsize=(21, 5.5)
        )

        for label, color, groups, intercept in (
            (f"C1 = {X}", "tab:blue", c1_groups, c1_intercept),
            (f"C2 = {Y}", "tab:orange", c2_groups, c2_intercept),
        ):
            agreements = [int(group["total_agreement"]) for group in groups]
            means = [float(group["mean_raw_logit"]) for group in groups]
            counts = [int(group["count"]) for group in groups]
            ax_agreement.scatter(
                agreements,
                means,
                s=[35 + 12 * count for count in counts],
                color=color,
                alpha=0.8,
                edgecolor="white",
                linewidth=0.8,
                label=f"{label}: mean raw logit (size = count)",
                zorder=3,
            )
            theory_x = list(range(K + 1))
            theory_y = [
                bayes_candidate_log_weight(value) + intercept for value in theory_x
            ]
            ax_agreement.plot(
                theory_x,
                theory_y,
                linestyle="--",
                color=color,
                alpha=0.9,
                label=f"{label}: Bayes slope + fitted intercept",
            )
            for agreement, mean, count in zip(agreements, means, counts):
                ax_agreement.annotate(
                    f"n={count}",
                    (agreement, mean),
                    xytext=(4, 5),
                    textcoords="offset points",
                    fontsize=8,
                    color=color,
                )

        ax_agreement.set(
            xlabel="Total agreement with candidate",
            ylabel="Mean raw candidate-token logit",
            title="Order-balanced candidate logits",
            xticks=range(K + 1),
        )
        ax_agreement.grid(alpha=0.25)
        ax_agreement.legend(fontsize=8)

        agreement_differences = [
            int(row["total_agreement_candidate_1"])
            - int(row["total_agreement_candidate_2"])
            for row in rows
        ]
        raw_logit_differences = [
            float(row["candidate_1_minus_candidate_2_raw_logit"])
            for row in rows
        ]
        ax_difference.scatter(
            agreement_differences,
            raw_logit_differences,
            s=42,
            color="tab:purple",
            alpha=0.55,
            edgecolor="none",
            label="one point per matched evidence scenario",
        )
        difference_grid = list(range(-K, K + 1))
        normative_log_odds = [
            difference * math.log(RELIABILITY / (1 - RELIABILITY))
            for difference in difference_grid
        ]
        ax_difference.plot(
            difference_grid,
            normative_log_odds,
            color="black",
            linestyle="--",
            label="exact Bayesian C1/C2 log-odds",
        )
        ax_difference.axhline(0, color="grey", linewidth=0.8)
        ax_difference.set(
            xlabel="total_agreement(C1) - total_agreement(C2)",
            ylabel="order-balanced raw logit(C1) - raw logit(C2)",
            title="Order-balanced semantic contrast",
            xticks=difference_grid,
        )
        ax_difference.grid(alpha=0.25)
        ax_difference.legend(fontsize=8)

        position_effects = [
            float(row["position_effect_on_candidate_difference"]) for row in rows
        ]
        ax_position.scatter(
            agreement_differences,
            position_effects,
            s=42,
            color="tab:green",
            alpha=0.55,
            edgecolor="none",
            label="half-contrast between presentation orders",
        )
        ax_position.axhline(0, color="black", linestyle="--", linewidth=0.9)
        ax_position.set(
            xlabel="total_agreement(C1) - total_agreement(C2)",
            ylabel="estimated first-position effect on C1-C2 logit contrast",
            title=f"Position effect (mean={fmean(position_effects):.3f})",
            xticks=difference_grid,
        )
        ax_position.grid(alpha=0.25)
        ax_position.legend(fontsize=8)

        fig.suptitle(f"{model_id} — reasoning budget {reasoning_budget}")
        fig.tight_layout()
        plt.show()


plot_budget_widget = reasoning_budget_selector("Reasoning budget")
plot_output = widgets.interactive_output(
    plot_agreement_logits, {"reasoning_budget": plot_budget_widget}
)
display(widgets.VBox([plot_budget_widget, plot_output]))

## Correlation with agreement as reasoning budget increases

For every reasoning budget and model, the following plot computes two correlations across the same 80 order-balanced evidence scenarios. The left axis is Spearman rank correlation and the right axis is Pearson correlation between $a(C_1)-a(C_2)$ and the order-balanced raw-logit contrast $\ell(C_1)-\ell(C_2)$ at the post-reasoning `ANSWER:` boundary.

In [ ]:
def tied_ranks(values: list[float | int]) -> list[float]:
    order = sorted(range(len(values)), key=values.__getitem__)
    ranks = [0.0] * len(values)
    start = 0
    while start < len(order):
        end = start + 1
        while end < len(order) and values[order[end]] == values[order[start]]:
            end += 1
        average_rank = (start + 1 + end) / 2
        for position in range(start, end):
            ranks[order[position]] = average_rank
        start = end
    return ranks


def safe_correlation(left: list[float], right: list[float]) -> float:
    if len(left) < 2 or len(set(left)) < 2 or len(set(right)) < 2:
        return float("nan")
    return correlation(left, right)


def correlations_by_reasoning_budget(
    rows: list[dict[str, object]],
) -> list[dict[str, float | int]]:
    values = []
    for reasoning_budget in REASONING_BUDGET_VALUES:
        selected = rows_at_reasoning_budget(rows, reasoning_budget)
        agreement_difference = [
            float(row["total_agreement_candidate_1"])
            - float(row["total_agreement_candidate_2"])
            for row in selected
        ]
        logit_difference = [
            float(row["candidate_1_minus_candidate_2_raw_logit"])
            for row in selected
        ]
        assert len(selected) == NUM_QUESTION_SETS * 2**K
        values.append({
            "reasoning_budget": reasoning_budget,
            "spearman": safe_correlation(
                tied_ranks(agreement_difference), tied_ranks(logit_difference)
            ),
            "pearson": safe_correlation(agreement_difference, logit_difference),
        })
    return values


correlation_rows_by_model = {
    model_id: correlations_by_reasoning_budget(rows)
    for model_id, rows in balanced_results_by_model.items()
}
fig, (ax_spearman, ax_pearson) = plt.subplots(1, 2, figsize=(15, 5.2), sharex=True)
for model_id, values in correlation_rows_by_model.items():
    budgets = [int(value["reasoning_budget"]) for value in values]
    ax_spearman.plot(
        budgets, [float(value["spearman"]) for value in values],
        marker="o", linewidth=1.6, label=model_id,
    )
    ax_pearson.plot(
        budgets, [float(value["pearson"]) for value in values],
        marker="o", linewidth=1.6, label=model_id,
    )
for axis, title, ylabel in (
    (ax_spearman, "Rank alignment across reasoning budgets", "Spearman correlation"),
    (ax_pearson, "Linear alignment at the ANSWER: boundary", "Pearson correlation"),
):
    axis.axhline(0, color="black", linewidth=0.8, alpha=0.6)
    axis.set(
        xlabel="XVsYPosteriorProbe reasoning budget",
        ylabel=ylabel,
        title=title,
        xticks=REASONING_BUDGET_VALUES,
        ylim=(-1.05, 1.05),
    )
    axis.tick_params(axis="x", rotation=45)
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
fig.tight_layout()
plt.show()
correlation_rows_by_model

## Raw wide truth tables

Each selected table has exactly $2^K$ rows—one per YES/NO answer pattern at one reasoning budget. For every one-based question-set index $j$ and question index $i$, it includes canonical $a_i^j(C_1)$ and $a_i^j(C_2)$ and their totals. It reports each candidate's raw logit under both presentation orders, the paired averages, the averaged semantic contrast, and the position-effect contrast. Multiple configured models receive separate tables. All model/budget tables are written as CSV; the widget displays only the selected budget.

In [ ]:
def raw_truth_table(
    rows: list[dict[str, object]],
) -> tuple[list[str], list[list[str]]]:
    by_key = {
        (int(row["answer_pattern_index"]), int(row["question_set_index"])): row
        for row in rows
    }
    headers = ["answer_pattern", *[f"report_i{i}" for i in range(1, K + 1)]]
    for j in range(1, NUM_QUESTION_SETS + 1):
        for i in range(1, K + 1):
            headers.extend([f"agreement_j{j}_i{i}_c1", f"agreement_j{j}_i{i}_c2"])
        headers.extend([
            f"agreement_j{j}_c1",
            f"agreement_j{j}_c2",
            f"raw_logit_j{j}_c1_order_c1_c2",
            f"raw_logit_j{j}_c1_order_c2_c1",
            f"raw_logit_j{j}_c2_order_c1_c2",
            f"raw_logit_j{j}_c2_order_c2_c1",
            f"raw_logit_j{j}_c1_balanced",
            f"raw_logit_j{j}_c2_balanced",
            f"raw_logit_j{j}_c1_minus_c2_balanced",
            f"position_effect_j{j}_on_c1_minus_c2",
        ])

    matrix = []
    for pattern_index, reports in enumerate(answer_patterns(K)):
        pattern = "".join("Y" if report == "YES" else "N" for report in reports)
        output_row = [pattern, *reports]
        for question_set_index in range(NUM_QUESTION_SETS):
            row = by_key[(pattern_index, question_set_index)]
            c1_flags = row["agreement_candidate_1_by_question"]
            c2_flags = row["agreement_candidate_2_by_question"]
            for c1_flag, c2_flag in zip(c1_flags, c2_flags):
                output_row.extend([str(int(c1_flag)), str(int(c2_flag))])
            output_row.extend([
                str(int(row["total_agreement_candidate_1"])),
                str(int(row["total_agreement_candidate_2"])),
                f"{float(row['candidate_1_raw_logit_order_c1_c2']):.8f}",
                f"{float(row['candidate_1_raw_logit_order_c2_c1']):.8f}",
                f"{float(row['candidate_2_raw_logit_order_c1_c2']):.8f}",
                f"{float(row['candidate_2_raw_logit_order_c2_c1']):.8f}",
                f"{float(row['candidate_1_raw_logit']):.8f}",
                f"{float(row['candidate_2_raw_logit']):.8f}",
                f"{float(row['candidate_1_minus_candidate_2_raw_logit']):.8f}",
                f"{float(row['position_effect_on_candidate_difference']):.8f}",
            ])
        matrix.append(output_row)
    assert len(matrix) == 2**K
    assert all(len(output_row) == len(headers) for output_row in matrix)
    return headers, matrix


def display_wide_table(headers: list[str], matrix: list[list[str]]) -> None:
    header_html = "".join(f"<th>{html.escape(value)}</th>" for value in headers)
    body_html = "".join(
        "<tr>" + "".join(f"<td>{html.escape(value)}</td>" for value in row) + "</tr>"
        for row in matrix
    )
    display(HTML(
        "<div style='overflow-x:auto; max-width:100%; max-height:650px'>"
        "<table style='border-collapse:collapse; font-size:11px; white-space:nowrap'>"
        "<thead><tr>" + header_html + "</tr></thead><tbody>" + body_html +
        "</tbody></table></div>"
        "<style>th,td{border:1px solid #bbb;padding:3px 6px;text-align:right}"
        "th{position:sticky;top:0;background:#eee;z-index:1}</style>"
    ))


truth_tables_started = monotonic()
log_progress("Building and exporting reasoning-budget truth tables")
truth_tables_by_model = {}
for model_id, all_rows in balanced_results_by_model.items():
    model_tables_started = monotonic()
    truth_tables_by_model[model_id] = {}
    for reasoning_budget in REASONING_BUDGET_VALUES:
        rows = rows_at_reasoning_budget(all_rows, reasoning_budget)
        headers, matrix = raw_truth_table(rows)
        truth_tables_by_model[model_id][reasoning_budget] = {
            "headers": headers, "rows": matrix
        }
        csv_path = (
            EXPERIMENT_ROOT / model_key(model_id)
            / f"agreement_truth_table_reasoning_{reasoning_budget:03d}.csv"
        )
        with csv_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle, lineterminator="\n")
            writer.writerow(headers)
            writer.writerows(matrix)
    log_progress(
        f"{model_id}: exported {len(REASONING_BUDGET_VALUES)} truth tables in "
        f"{monotonic() - model_tables_started:.1f} s"
    )
log_progress(
    f"Truth-table build complete in {monotonic() - truth_tables_started:.1f} s"
)


def display_truth_tables(reasoning_budget: int) -> None:
    for model_id, tables_by_budget in truth_tables_by_model.items():
        table = tables_by_budget[reasoning_budget]
        print(
            f"{model_id} — reasoning budget {reasoning_budget}: "
            f"{len(table['rows'])} x {len(table['headers'])}"
        )
        display_wide_table(table["headers"], table["rows"])


truth_table_budget_widget = reasoning_budget_selector("Reasoning budget")
truth_table_output = widgets.interactive_output(
    display_truth_tables, {"reasoning_budget": truth_table_budget_widget}
)
display(widgets.VBox([truth_table_budget_widget, truth_table_output]))

## Compact audit view

This final long-form sample makes the mapping from question membership and observed report to each canonical agreement bit easy to inspect, then shows both order-specific logits and their paired summaries for the selected reasoning budget.

In [ ]:
def display_audit_view(reasoning_budget: int) -> None:
    for model_id, all_rows in balanced_results_by_model.items():
        sample = rows_at_reasoning_budget(all_rows, reasoning_budget)[0]
        print(f"{model_id} — reasoning budget {reasoning_budget}")
        for i, (membership, report, c1_agrees, c2_agrees) in enumerate(zip(
            sample["membership_sets"],
            sample["observed_reports"],
            sample["agreement_candidate_1_by_question"],
            sample["agreement_candidate_2_by_question"],
        ), start=1):
            print({
                "question_i": i,
                "membership_set": membership,
                "observed_report": report,
                "agreement_i_c1": c1_agrees,
                "agreement_i_c2": c2_agrees,
            })
        print({
            "reasoning_budget": sample["reasoning_budget"],
            "total_agreement_c1": sample["total_agreement_candidate_1"],
            "total_agreement_c2": sample["total_agreement_candidate_2"],
            "c1_logit_order_c1_c2": sample["candidate_1_raw_logit_order_c1_c2"],
            "c1_logit_order_c2_c1": sample["candidate_1_raw_logit_order_c2_c1"],
            "c2_logit_order_c1_c2": sample["candidate_2_raw_logit_order_c1_c2"],
            "c2_logit_order_c2_c1": sample["candidate_2_raw_logit_order_c2_c1"],
            "c1_raw_logit_balanced": sample["candidate_1_raw_logit"],
            "c2_raw_logit_balanced": sample["candidate_2_raw_logit"],
            "c1_minus_c2_balanced": sample["candidate_1_minus_candidate_2_raw_logit"],
            "position_effect": sample["position_effect_on_candidate_difference"],
            "presentation_artifacts": sample["presentation_artifacts"],
        })


audit_budget_widget = reasoning_budget_selector("Reasoning budget")
audit_output = widgets.interactive_output(
    display_audit_view, {"reasoning_budget": audit_budget_widget}
)
display(widgets.VBox([audit_budget_widget, audit_output]))